# Candidate library audit

This notebook is part of the staged manuscript-reproduction workflow. During automated validation it resolves placeholder paths to the deterministic demo dataset; once local real-data paths are configured it will run against the real case-study artifacts.

In [ ]:
import os
import sys
from pathlib import Path


def _find_repo_root() -> Path:
    env_root = os.environ.get("BSM_PUBLIC_RF_REPO_ROOT")
    if env_root:
        candidate = Path(env_root).resolve()
        if (candidate / "src").exists() and (candidate / "pyproject.toml").exists():
            return candidate

    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "src").exists() and (candidate / "pyproject.toml").exists():
            return candidate

    raise RuntimeError("Could not locate the bsm-public-rf repository root.")


REPO_ROOT = _find_repo_root()
SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

In [ ]:
from bsm_rfm import build_manuscript_notebook_context

context = build_manuscript_notebook_context(REPO_ROOT, "01_candidate_library_audit.ipynb")
feature_catalog = context.tables["manuscript_feature_catalog"]
feature_catalog.groupby("feature_type").size().rename("count").reset_index()

### Step 1: Load Features and Candidate Library

Load the feature DataFrame and candidate library definitions. Prepare validation checks.

**Variables:**
- `dfm_full`: Feature DataFrame (n_samples, ~350 features) from stage 00
- `candidates`: Candidate library metadata (feature definitions, groups, descriptions)
- `audit_checks`: List of validation rules (variance, missing data, duplicates, etc.)

### Step 2: Run Validation Checks

Apply all QA checks to identify issues. Exclude features that fail criteria.

**Operations:**
- Check variance against threshold
- Count missing data rate
- Detect exact duplicates and high-correlation pairs
- Flag features matching exclusion patterns (e.g., all-zero, identifiers)

**Output:**
- `audit_results`: DataFrame with pass/fail for each check and feature